# Imports

In [2]:
from Functions.Fcts_FE import make_experiment, apply_threshold_changes, test_skeletonization, estimate_staining_thresholds_all_rounds, plot_thresholds_multicycle, extract_features_multicycle
from Functions.Fcts_Base import get_stainings, find_zarr_dirs, extract_ome_zarr_tables
from Functions.Fcts_Plotting import segmentation_fidelity_check

%load_ext autoreload
%autoreload 2

# User Input

### General Settings

Provide information regarding directories, used barcodes, and define the file name for results.
Setup .xls file must be in source folder. See GitHub readme for more information.

In [ ]:
source = "YOUR PATH TO EXPERIMENT FOLDER"   # Point to experiment folder. Has to include setup .xls file.

analysis_dir = None                         # Optional: Analysis directory. If None this will default to the source folder. Creates 4 folder substructure for standardization in subsequent notebooks.

file_ending = ".zarr"                       # File ending to search for ome_zarr files.

experiment_ID = "YOUR EXPERIMENT ID" 

### Measurement settings

In [ ]:
quantiles_to_calc = [0.01, 0.25, 0.50, 0.75, 0.99]  # List of quantiled to be calculated during feature extraction for every staining.

pyramid_level = 0                                   # Fractal pyramid level to use for feature extraction.

table_name = "organoid_ROI_table"                   # Fractal ROI table name to load images.

label_name = "organoids"                            # Fractal label name to load segmentation.

# Multicycle settings
segmentation_round = 0                              # Round that contains ROI table + labels. In most cases this will be the first round (0).
rounds_to_extract = [0]                           # Imaging rounds to extract (e.g. [0, 1, 2]). If no multicycle, set to [0].

do_moments_features = True
do_skeleton_features = True
do_thresholded_features = True
do_substructure_features = True

# Load files and display experimental setup
Loads information from .xls setup files to subsequently link them to features of extracted organoids. Will display setup of all found imaging plates, with each well showing information regarding its conditions in the format of [Medium condition, used antibody mix, used cell line, additional condition]

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
stainings = get_stainings(source)

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
folder, analysis_dir = find_zarr_dirs(source, file_ending = file_ending, analysis_dir = analysis_dir) # Find ome_zarr files in source folder.
experiment_setup, barcodes = make_experiment(source) # Create experiment setup from Layout in source folder.

# Load ome-zarr plates
Uses ez-zarr to load ome-zarr plates as objects for further processing

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
ome_zarr_dict, ome_zarr_df = extract_ome_zarr_tables(experiment_setup, source, folder, table_name)

# Estimate thresholds

### Automatic estimation (per round)
Runs automatic estimation for every round in rounds_to_extract.

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
control_condition = None  # Must be key in Medium tab of .xls setup file. Set to None if not applicable.
n = 5                     # Number of organoids picked randomly per timepoint
seed = 0                  # Random seed
sigma = 1                 # Sigma for gaussian blurring
q = 0.5                   # Quantile used to pick final threshold

# ── Run Cell ────────────────────────────────────────────────────────────────────────
thresholds_raw_by_round, thresholds_by_round, dict_org_by_round, timepoints_by_round = (
    estimate_staining_thresholds_all_rounds(
        rounds=rounds_to_extract,
        ome_zarr_df=ome_zarr_df,
        ome_zarr_dict=ome_zarr_dict,
        stainings=stainings,
        experiment_setup=experiment_setup,
        table_name=table_name,
        label_name=label_name,
        pyramid_level=pyramid_level,
        segmentation_round=segmentation_round,
        control_condition=control_condition,
        n=n,
        seed=seed,
        sigma=sigma,
        q=q,
    )
)

### Manual change
Optional manual change of non-fitting automatic thresholds (per round).

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
thresholds_to_change = {
    "R0__DAPI": 1000,
    "R0__KRT7": 500,
    "R1__EPCAM": 750,
}

# ── Run Cell ────────────────────────────────────────────────────────────────────────
thresholds_raw_by_round, thresholds_by_round = apply_threshold_changes(
    thresholds_raw_by_round,
    thresholds_by_round,
    thresholds_to_change,
)

### Test thresholds
Plots the distribution of thresholds found per staining (for one selected round).

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
round_to_plot = 0 # Round to plot thresholds for.

# ── Run Cell ────────────────────────────────────────────────────────────────────────
plot_thresholds_multicycle(
    thresholds = thresholds_raw_by_round[round_to_plot],
    dict_org = dict_org_by_round[round_to_plot],
    timepoints_lst = timepoints_by_round[round_to_plot],
    ome_zarr_dict = ome_zarr_dict,
    table_name = table_name,
    label_name = label_name,
    seed = 0,
    pyramid_level = pyramid_level,
    round_id = round_to_plot,
    segmentation_round = segmentation_round,
)

# Visualize segmentation fidelity
Loads n random wells of every plate and overlays it with its segmentation mask to check for segmentation fidelity.

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
channel = 0                 # Channel to plot. Related to order of stainings in "stainings" dictionary. Round 0 only.
channel_color = "inferno"   # Color of channel to plot.
channel_range = [100,4000]  # Range of values to plot for channel. Must
n = 5                       # Number of wells to plot per timepoint.
pyramid_lvl_plot = 2        # OME_zarr Pyramid level to plot.

# ── Run Cell ────────────────────────────────────────────────────────────────────────
segmentation_fidelity_check(
    ome_zarr_dict,
    channel = channel,
    channel_color = channel_color,
    channel_range = channel_range,
    n = n,
    label_name = label_name,
    alpha = 0.5,
    pyramid_lvl_plot = pyriamid_lvl_plot,
)

# Test of crypt features extraction

Used to test parameters important to extract crypt features, such as sigma_skeleton, n_angle_determination, and radius_multiplier. Set parameters will be used in a subsequent step for feature extraction. Will show found crypts as well as an approximate for the non-crpyt region marked by a circle on n random organoids per plate.

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
sigma_skeleton = 8              # Integer. Sigma for gaussian blurring of mask before skeletonization. Sharp segmentation edges may lead to difficulties, necessating higher sigma_skeleton numbers.
n_angle_determination = 50      # Integer. Number of pixel used at the end of each branch to estimate its direction until the segmentation border. If the number is higher than the length of the branch, the whole branch will be used instead.
radius_multiplier = 0.8         # Float. Multiplier of maximum-inscribed-circle radius to estimate crypt-free region. The lower the number, the more sensititve the algorithm is to smaller crypts.
n = 10                          # Integer. Number of randomly chosen organoids per imaging plate.
seed = 0                        # Integer. Random seed.

# ── Run Cell ────────────────────────────────────────────────────────────────────────
test_skeletonization(
    barcodes,
    stainings,
    experiment_setup,
    ome_zarr_dict,
    ome_zarr_df,
    table_name = table_name,
    label_name = label_name,
    pyramid_level = pyramid_level,
    n = n,
    seed = seed,
    sigma_skeleton = sigma_skeleton,
    n_angle_determination = n_angle_determination,
    radius_multiplier = radius_multiplier
)

# Extract Features

Function extracts morphological and staining features from all segmented organoids within the source+folder structure. Results will be saved in an .csv and .h5ad file in the analysis_dir+\"2_Results\"+result_file_name. .h5ad output of this function should be used as an input for the subsequent notebook \"2_Filtering\". 

In [ ]:
# ── Run Cell ────────────────────────────────────────────────────────────────────────
ad = extract_features_multicycle(
    ome_zarrs_dict = ome_zarr_dict,
    table_name = table_name,
    label_name = label_name,
    pyramid_level = pyramid_level,
    source = source,
    folder = folder,
    analysis_dir = analysis_dir,
    barcodes = barcodes,
    experiment_setup = experiment_setup,
    thresholds = thresholds_by_round,
    stainings = stainings,
    result_file_name = "1_FeatureExtraction",
    radius_multiplier = radius_multiplier,
    sigma_skeleton = sigma_skeleton,
    quantiles_to_calc = quantiles_to_calc,
    experiment_ID = experiment_ID,
    rounds_to_extract = rounds_to_extract,
    segmentation_round = segmentation_round,
    compute_cross_round_pearson = True,
    do_moments_features = do_moments_features,
    do_skeleton_features = do_skeleton_features,
    do_thresholded_features = do_thresholded_features,
    do_substructure_features = do_substructure_features,
)